# 05 — Analyse post-entraînement

Lit les `logs/<run_id>/history.jsonl` produits par [`src/train.py`](../src/train.py) et trace les courbes utiles pour **interpréter l'efficacité d'un run et repérer un overfitting**.

Indépendant de wandb : tout est reconstruit à partir des JSONL locaux.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from pathlib import Path
import matplotlib.pyplot as plt

from src import config as cfg
from src.logging_utils import read_history, metrics_records

LOGS_ROOT = Path(cfg.LOGS)
print("Logs root :", LOGS_ROOT)

available_runs = sorted(
    [p.name for p in LOGS_ROOT.iterdir() if (p / "history.jsonl").is_file()]
) if LOGS_ROOT.exists() else []
print("Runs disponibles :")
for r in available_runs:
    print("  -", r)

## 1. Sélection du run

In [ ]:
RUN_ID = available_runs[-1] if available_runs else "mini-cpu-demo"
print("RUN_ID :", RUN_ID)

records = read_history(LOGS_ROOT / RUN_ID)
mrecs = metrics_records(records)
print(f"{len(mrecs)} lignes de métriques.")

def col(key):
    return [r.get(key) for r in mrecs]

epoch       = col("epoch")
train_loss  = col("train_loss")
val_loss    = col("val_loss")
val_si_sdr  = col("val_si_sdr")
lr          = col("lr")
gap         = col("gap")
gap_ratio   = col("gap_ratio")
plateau     = col("val_plateau_epochs")
val_inc     = col("val_increasing")
diverging   = col("diverging")
mask_mean   = col("mask_mean")
mask_std    = col("mask_std")

## 2. Courbes principales (loss + SI-SDR)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epoch, train_loss, label="train")
axes[0].plot(epoch, val_loss,   label="val")
axes[0].set_title("Loss MSE (linéaire)"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_xlabel("epoch")

axes[1].plot(epoch, train_loss, label="train")
axes[1].plot(epoch, val_loss,   label="val")
axes[1].set_yscale("log")
axes[1].set_title("Loss MSE (log)"); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xlabel("epoch")

axes[2].plot(epoch, val_si_sdr, color="tab:green")
axes[2].set_title("SI-SDR validation (dB)"); axes[2].grid(alpha=0.3)
axes[2].set_xlabel("epoch")

fig.tight_layout(); plt.show()

## 3. Indicateurs d'overfitting

- **gap = val_loss − train_loss** : doit rester petit et stable. S'il grandit régulièrement → overfit.
- **gap_ratio = val_loss / train_loss** : autour de 1 quand tout va bien, > 2 = overfit franc.
- **val_plateau_epochs** : nombre d'epochs sans amélioration de val_loss.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epoch, gap, color="tab:red")
axes[0].axhline(0, color="k", linestyle="--", linewidth=0.5)
axes[0].set_title("gap = val − train"); axes[0].grid(alpha=0.3)
axes[0].set_xlabel("epoch")

axes[1].plot(epoch, gap_ratio, color="tab:purple")
axes[1].axhline(1, color="k", linestyle="--", linewidth=0.5)
axes[1].set_title("gap_ratio = val / train"); axes[1].grid(alpha=0.3)
axes[1].set_xlabel("epoch")

axes[2].plot(epoch, plateau, color="tab:gray")
axes[2].set_title("epochs sans amélioration de val"); axes[2].grid(alpha=0.3)
axes[2].set_xlabel("epoch")

fig.tight_layout(); plt.show()

## 4. Statistiques du masque appris

- **mask_mean** : si très proche de 1 → le réseau "laisse tout passer" (il ne débruite pas). Si très proche de 0 → il coupe tout (sortie quasi muette).
- **mask_std** : trop bas → le masque est uniforme, le réseau n'a rien appris de discriminant.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epoch, mask_mean); axes[0].set_title("mask_mean"); axes[0].grid(alpha=0.3)
axes[0].set_xlabel("epoch"); axes[0].set_ylim(0, 1)
axes[1].plot(epoch, mask_std, color="tab:orange"); axes[1].set_title("mask_std"); axes[1].grid(alpha=0.3)
axes[1].set_xlabel("epoch")
fig.tight_layout(); plt.show()

## 5. Tableau récapitulatif des flags par epoch

In [ ]:
print(f"{'ep':>3} {'train':>8} {'val':>8} {'SI-SDR':>8} {'gap':>7} {'plat':>5} {'inc':>4} {'div':>4}")
for i in range(len(epoch)):
    print(
        f"{epoch[i]:>3} "
        f"{train_loss[i]:>8.4f} {val_loss[i]:>8.4f} "
        f"{val_si_sdr[i]:>+8.2f} {gap[i]:>+7.4f} "
        f"{plateau[i]:>5d} "
        f"{'Y' if val_inc[i] else '.':>4} "
        f"{'Y' if diverging[i] else '.':>4}"
    )

## 6. Comparaison de plusieurs runs

Utile en semaine 4 quand on testera plusieurs variantes (loss alternative, modèle plus profond, etc.).

In [ ]:
RUNS_TO_COMPARE = available_runs   # éditer pour ne garder que ceux à comparer

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for run in RUNS_TO_COMPARE:
    recs = metrics_records(read_history(LOGS_ROOT / run))
    if not recs:
        continue
    ep = [r["epoch"] for r in recs]
    axes[0].plot(ep, [r["val_loss"] for r in recs], label=run)
    axes[1].plot(ep, [r["val_si_sdr"] for r in recs], label=run)

axes[0].set_title("val_loss"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3); axes[0].set_xlabel("epoch")
axes[1].set_title("SI-SDR val (dB)"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3); axes[1].set_xlabel("epoch")
fig.tight_layout(); plt.show()